# RSI-Plateau: Kaggle Pilot (Free Tier, H1 + Controls)

Runs the reduced pilot matrix on a free T4: **oracle vs weak judge** (H1) plus
controls (random-filter, fixed-data) and the ReST ablation. Each job is one
self-training run (~1-2 GPU-hours). Completed jobs are **skipped on re-run**,
and every round checkpoints to `result.json`, so session drops cost nothing.

### Before you run
1. Right sidebar -> Settings: **Accelerator GPU T4 x2**, **Internet On**.
2. For anything over ~30 min, use **Save Version -> Run All (background)**
   instead of interactive Run (12h limit, survives tab closes).

### Plan (fits ~30 GPU-hrs/week)
- **Week 1:** `oracle`, `weak` x 2 seeds = 4 jobs (the H1 headline)
- **Week 2:** `random_filter`, `fixed_data`, `rest` x 2 seeds = 6 jobs

To accumulate across weeks: download `artifacts/pilot/` from week 1, then
upload it as a Kaggle **dataset** and attach it as **Input** next week —
the driver copies prior results in and skips finished jobs automatically.
See the restore cell below.

In [ ]:
# 1. GPU check
!nvidia-smi

In [ ]:
# 2. Fresh clone (pins the fixed code)
import os
os.chdir("/kaggle/working")
REPO = "https://github.com/AbhiGuru25/Recursive-self-improvement-RSI.git"
if not os.path.isdir("/kaggle/working/RSI"):
    !git clone -q $REPO /kaggle/working/RSI
%cd /kaggle/working/RSI
!git pull -q
!git log --oneline -3

In [ ]:
# 3. Install (protect Kaggle's CUDA torch; drop broken preinstalled torchao)
!pip install -q -e . --no-deps
!pip install -q peft trl accelerate sentence-transformers
!pip uninstall -y torchao

In [ ]:
# 4. OPTIONAL restore: copy week-1 results in from an attached Input dataset
#    (Dataset path looks like /kaggle/input/<your-dataset>/pilot/).
#    Skip this cell on week 1.
import glob, pathlib, shutil
prior = glob.glob("/kaggle/input/*/pilot")
if prior:
    src = pathlib.Path(prior[0])
    dst = pathlib.Path("artifacts/pilot")
    for job_dir in src.iterdir():
        if job_dir.is_dir() and not (dst / job_dir.name).exists():
            shutil.copytree(job_dir, dst / job_dir.name)
    print(f"restored prior results from {src}")
else:
    print("no prior results attached; starting fresh")

In [ ]:
# 5. WEEK 1 jobs: H1 headline (oracle vs weak judge), 2 seeds each.
#    Swap the list for week 2: random_filter fixed_data rest
!CUDA_VISIBLE_DEVICES=0 python -u scripts/run_pilot.py --jobs oracle weak --seeds 2 2>&1 | tail -30

In [ ]:
# 6. Aggregate what finished (works on partial results too)
!python scripts/aggregate_results.py --root artifacts/pilot --figures 2>&1 | tail -20

## Reading the output

- `artifacts/pilot/<job>_s<seed>/result.json` — one per finished job.
- `artifacts/aggregate/summary.json` + `accuracy_trajectories.png` — H1 plot.
- **H1 supported** if the oracle curve climbs past the round where the weak-judge
  curve plateaus, at matched acceptance rate (the driver enables α-matching).
- Download `artifacts/` from Output (`/kaggle/working`) before the session ends,
  or commit (Save Version) so outputs persist.